# 📱 Deteksi Kecanduan Smartphone
## Notebook 5: Hybrid CNN + XGBoost
---
**Arsitektur:** CNN digunakan sebagai *feature extractor* dari data tabular  
(fitur numerik direshape menjadi pseudo-1D signal), kemudian fitur yang dihasilkan  
lapisan tersembunyi CNN diumpankan ke XGBoost sebagai classifier akhir.

```
Input Tabular → [Reshape 1D] → CNN (Conv1D + Pooling) → Flatten
                                                            ↓
                                              Deep Features (CNN Output)
                                                            ↓
                                                    XGBoost Classifier
                                                            ↓
                                               Prediksi Kelas (1–5)
```

Input  : `X_train.csv`, `y_train.csv`, `X_test.csv`, `y_test.csv`  
Output : `cnn_feature_extractor.h5`, `hybrid_cnn_xgb_model.pkl`

### 5.1 Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.utils import to_categorical

# Machine Learning
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)
from sklearn.preprocessing import label_binarize
from sklearn.inspection import permutation_importance
import shap

# Seed untuk reprodusibilitas
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print(f'✅ TensorFlow  : {tf.__version__}')
print(f'✅ XGBoost     : {xgb.__version__}')

### 5.2 Load Data

In [ ]:
X_train = pd.read_csv('X_train.csv')
X_test  = pd.read_csv('X_test.csv')
y_train = pd.read_csv('y_train.csv').squeeze()
y_test  = pd.read_csv('y_test.csv').squeeze()

N_CLASSES  = 5
N_FEATURES = X_train.shape[1]

# Label: 0-indexed untuk XGBoost & CNN
y_train_0 = (y_train - 1).values
y_test_0  = (y_test  - 1).values

# One-hot untuk CNN (training CNN sebagai autoencoder fitur)
y_train_oh = to_categorical(y_train_0, num_classes=N_CLASSES)
y_test_oh  = to_categorical(y_test_0,  num_classes=N_CLASSES)

print(f'X_train : {X_train.shape}   y_train : {y_train_0.shape}')
print(f'X_test  : {X_test.shape}    y_test  : {y_test_0.shape}')
print(f'Fitur   : {N_FEATURES}   Kelas   : {N_CLASSES}')

### 5.3 Reshape Data untuk CNN (1D Convolution)

> Data tabular direshape menjadi `(samples, n_features, 1)` agar bisa diproses  
> oleh Conv1D — setiap fitur diperlakukan sebagai satu "time step".

In [ ]:
X_train_cnn = X_train.values.reshape(-1, N_FEATURES, 1)
X_test_cnn  = X_test.values.reshape(-1, N_FEATURES, 1)

print(f'Shape X_train untuk CNN : {X_train_cnn.shape}  → (samples, features, channels)')
print(f'Shape X_test  untuk CNN : {X_test_cnn.shape}')

### 5.4 Arsitektur CNN Feature Extractor

Lapisan CNN dilatih sebagai *supervised classifier*, lalu output dari  
lapisan `GlobalAveragePooling` (sebelum output layer) diambil sebagai  
**deep feature vector** untuk XGBoost.

In [ ]:
def build_cnn(input_shape, n_classes):
    """
    Arsitektur CNN 1D untuk data tabular.
    
    Block 1: Conv1D(64) → BatchNorm → ReLU → MaxPool
    Block 2: Conv1D(128) → BatchNorm → ReLU → MaxPool
    Block 3: Conv1D(256) → BatchNorm → ReLU → GlobalAvgPool  ← Feature Vector
    Head   : Dense(128) → Dropout → Dense(n_classes, softmax)
    """
    inputs = layers.Input(shape=input_shape, name='input_features')

    # ── Block 1 ──────────────────────────────────────
    x = layers.Conv1D(64, kernel_size=3, padding='same', name='conv1')(inputs)
    x = layers.BatchNormalization(name='bn1')(x)
    x = layers.Activation('relu', name='relu1')(x)
    x = layers.MaxPooling1D(pool_size=2, padding='same', name='pool1')(x)
    x = layers.Dropout(0.2, name='drop1')(x)

    # ── Block 2 ──────────────────────────────────────
    x = layers.Conv1D(128, kernel_size=3, padding='same', name='conv2')(x)
    x = layers.BatchNormalization(name='bn2')(x)
    x = layers.Activation('relu', name='relu2')(x)
    x = layers.MaxPooling1D(pool_size=2, padding='same', name='pool2')(x)
    x = layers.Dropout(0.2, name='drop2')(x)

    # ── Block 3 ──────────────────────────────────────
    x = layers.Conv1D(256, kernel_size=3, padding='same', name='conv3')(x)
    x = layers.BatchNormalization(name='bn3')(x)
    x = layers.Activation('relu', name='relu3')(x)

    # ── Feature Extraction Layer ──────────────────────
    features = layers.GlobalAveragePooling1D(name='gap_features')(x)  # ← Deep features

    # ── Classification Head ───────────────────────────
    x = layers.Dense(128, activation='relu', name='dense1')(features)
    x = layers.Dropout(0.3, name='drop_head')(x)
    outputs = layers.Dense(n_classes, activation='softmax', name='output')(x)

    model = models.Model(inputs=inputs, outputs=outputs, name='CNN_Feature_Extractor')
    return model


cnn_model = build_cnn(input_shape=(N_FEATURES, 1), n_classes=N_CLASSES)

cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

### 5.5 Visualisasi Arsitektur CNN

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

blocks = [
    (1.0,  7.5, '#3498db', 'INPUT\n(samples, features, 1)',  ''),
    (1.0,  6.0, '#2980b9', 'Conv1D(64) + BatchNorm\n+ ReLU + MaxPool + Dropout', 'Block 1'),
    (1.0,  4.5, '#1abc9c', 'Conv1D(128) + BatchNorm\n+ ReLU + MaxPool + Dropout', 'Block 2'),
    (1.0,  3.0, '#16a085', 'Conv1D(256) + BatchNorm\n+ ReLU', 'Block 3'),
    (1.0,  1.8, '#e67e22', 'GlobalAvgPool1D\n→ Deep Feature Vector (256-dim)', '🔑 Feature Layer'),
    (5.5,  1.8, '#8e44ad', 'Dense(128) + Dropout(0.3)',  'XGBoost Input'),
    (5.5,  0.5, '#e74c3c', 'XGBoost Classifier\n(5 Kelas Kecanduan)', 'Prediction'),
]

for x, y, color, label, tag in blocks:
    rect = plt.Rectangle((x, y-0.5), 3.5, 0.9,
                          facecolor=color, alpha=0.85, edgecolor='white', linewidth=2)
    ax.add_patch(rect)
    ax.text(x + 1.75, y, label, ha='center', va='center',
            fontsize=8.5, color='white', fontweight='bold')
    if tag:
        ax.text(x + 3.6, y, tag, ha='left', va='center',
                fontsize=7.5, color='gray', style='italic')

# Arrows CNN stack
for y_start, y_end in [(7.0, 6.5), (5.5, 5.0), (4.0, 3.5), (2.8, 2.3)]:
    ax.annotate('', xy=(2.75, y_end), xytext=(2.75, y_start),
                arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=2))

# Arrow ke XGBoost
ax.annotate('', xy=(5.5, 1.8), xytext=(4.5, 1.8),
            arrowprops=dict(arrowstyle='->', color='#e67e22', lw=2.5))
ax.annotate('', xy=(7.25, 1.0), xytext=(7.25, 1.3),
            arrowprops=dict(arrowstyle='->', color='#8e44ad', lw=2))

ax.set_title('Arsitektur Hybrid CNN + XGBoost\nDeteksi Kecanduan Smartphone',
             fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('05_arsitektur_hybrid.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Diagram arsitektur disimpan.')

### 5.6 Training CNN

In [ ]:
# Callbacks
cb_early = callbacks.EarlyStopping(
    monitor='val_loss', patience=15,
    restore_best_weights=True, verbose=1
)
cb_reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5,
    patience=7, min_lr=1e-6, verbose=1
)
cb_checkpoint = callbacks.ModelCheckpoint(
    'cnn_best_weights.h5', monitor='val_accuracy',
    save_best_only=True, verbose=0
)

print('🚀 Memulai training CNN...')
history = cnn_model.fit(
    X_train_cnn, y_train_oh,
    validation_data=(X_test_cnn, y_test_oh),
    epochs=100,
    batch_size=32,
    callbacks=[cb_early, cb_reduce_lr, cb_checkpoint],
    verbose=1
)

# Evaluasi CNN murni
cnn_loss, cnn_acc = cnn_model.evaluate(X_test_cnn, y_test_oh, verbose=0)
print(f'\n✅ CNN Accuracy (standalone): {cnn_acc:.4f} ({cnn_acc*100:.2f}%)')

### 5.7 Learning Curve CNN

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss
axes[0].plot(history.history['loss'],     label='Train Loss',    color='#3498db', lw=2)
axes[0].plot(history.history['val_loss'], label='Val Loss',      color='#e74c3c', lw=2, ls='--')
axes[0].set_title('CNN Training – Loss', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Categorical Cross-Entropy')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(history.history['accuracy'],     label='Train Accuracy', color='#2ecc71', lw=2)
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy',   color='#e67e22', lw=2, ls='--')
axes[1].set_title('CNN Training – Accuracy', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Learning Curve – CNN Feature Extractor', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('05_cnn_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Learning curve CNN disimpan.')

### 5.8 Ekstraksi Deep Features dari CNN

Ambil output dari lapisan `gap_features` (sebelum classification head)  
sebagai representasi fitur berdimensi tinggi untuk XGBoost.

In [ ]:
# Buat model ekstraksi fitur (input → GlobalAvgPool layer)
feature_extractor = models.Model(
    inputs=cnn_model.input,
    outputs=cnn_model.get_layer('gap_features').output,
    name='CNN_FeatureExtractor'
)

# Ekstrak fitur dari CNN
X_train_deep = feature_extractor.predict(X_train_cnn, verbose=0)
X_test_deep  = feature_extractor.predict(X_test_cnn,  verbose=0)

print(f'✅ Deep features diekstrak:')
print(f'   X_train_deep shape : {X_train_deep.shape}   (samples × 256 CNN features)')
print(f'   X_test_deep  shape : {X_test_deep.shape}')

# Gabungkan deep features CNN + fitur asli tabular (Late Fusion)
X_train_hybrid = np.hstack([X_train_deep, X_train.values])
X_test_hybrid  = np.hstack([X_test_deep,  X_test.values])

print(f'\n✅ Hybrid features (CNN + Tabular):')
print(f'   X_train_hybrid shape : {X_train_hybrid.shape}')
print(f'   X_test_hybrid  shape : {X_test_hybrid.shape}')

### 5.9 Visualisasi t-SNE Deep Features

In [ ]:
from sklearn.manifold import TSNE

print('🔄 Menjalankan t-SNE pada deep features (mungkin 1–2 menit)...')
tsne = TSNE(n_components=2, random_state=SEED, perplexity=40, n_iter=1000)
X_tsne = tsne.fit_transform(X_test_deep)

colors_map = {0:'#2ecc71', 1:'#3498db', 2:'#f39c12', 3:'#e74c3c', 4:'#8e44ad'}
class_labels = {0:'Sangat Rendah',1:'Rendah',2:'Sedang',3:'Tinggi',4:'Sangat Tinggi'}

plt.figure(figsize=(11, 8))
for cls in range(5):
    mask = y_test_0 == cls
    plt.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                c=colors_map[cls], label=f'Kelas {cls+1}: {class_labels[cls]}',
                alpha=0.75, s=50, edgecolors='white', linewidths=0.4)

plt.title('t-SNE Visualisasi Deep Features CNN\n(Ruang Fitur Setelah Feature Extraction)',
          fontweight='bold', fontsize=13)
plt.xlabel('t-SNE Dimensi 1')
plt.ylabel('t-SNE Dimensi 2')
plt.legend(loc='upper right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('05_tsne_deep_features.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ t-SNE plot disimpan.')

### 5.10 Training XGBoost pada Hybrid Features

In [ ]:
# Hyperparameter tuning untuk XGBoost Hybrid
param_grid = {
    'n_estimators'    : [100, 200, 300],
    'max_depth'       : [3, 5, 7],
    'learning_rate'   : [0.05, 0.1, 0.2],
    'subsample'       : [0.7, 0.9],
    'colsample_bytree': [0.7, 0.9],
    'min_child_weight': [1, 3]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

xgb_hybrid_base = XGBClassifier(
    objective='multi:softmax',
    num_class=N_CLASSES,
    random_state=SEED,
    eval_metric='mlogloss',
    use_label_encoder=False,
    verbosity=0
)

random_search_hybrid = RandomizedSearchCV(
    estimator=xgb_hybrid_base,
    param_distributions=param_grid,
    n_iter=30,
    scoring='accuracy',
    cv=cv,
    random_state=SEED,
    n_jobs=-1,
    verbose=1
)

print('🔍 Hyperparameter Tuning XGBoost pada Hybrid Features...')
random_search_hybrid.fit(X_train_hybrid, y_train_0)

print(f'\n✅ Best CV Accuracy (Hybrid) : {random_search_hybrid.best_score_:.4f}')
print(f'Best Parameters             : {random_search_hybrid.best_params_}')

In [ ]:
# Training model hybrid terbaik
best_params_hybrid = random_search_hybrid.best_params_

xgb_hybrid = XGBClassifier(
    objective='multi:softmax',
    num_class=N_CLASSES,
    random_state=SEED,
    eval_metric='mlogloss',
    use_label_encoder=False,
    verbosity=0,
    **best_params_hybrid
)

eval_set = [(X_train_hybrid, y_train_0), (X_test_hybrid, y_test_0)]
xgb_hybrid.fit(X_train_hybrid, y_train_0, eval_set=eval_set, verbose=False)

y_pred_hybrid = xgb_hybrid.predict(X_test_hybrid)
acc_hybrid    = accuracy_score(y_test_0, y_pred_hybrid)

print(f'✅ Hybrid CNN+XGBoost Accuracy : {acc_hybrid:.4f} ({acc_hybrid*100:.2f}%)')

### 5.11 Learning Curve XGBoost Hybrid

In [ ]:
results_hybrid = xgb_hybrid.evals_result()
epochs_hybrid  = len(results_hybrid['validation_0']['mlogloss'])

plt.figure(figsize=(10, 5))
plt.plot(range(epochs_hybrid), results_hybrid['validation_0']['mlogloss'],
         label='Train Loss', color='#8e44ad', lw=2)
plt.plot(range(epochs_hybrid), results_hybrid['validation_1']['mlogloss'],
         label='Test Loss',  color='#e74c3c', lw=2, ls='--')
plt.xlabel('Epoch (n_estimators)')
plt.ylabel('Log Loss')
plt.title('Learning Curve – XGBoost pada Hybrid CNN Features', fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('05_xgb_hybrid_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.12 Evaluasi Lengkap – Hybrid CNN + XGBoost

In [ ]:
y_pred_proba_hybrid = xgb_hybrid.predict_proba(X_test_hybrid)

acc_h  = accuracy_score(y_test_0, y_pred_hybrid)
prec_h = precision_score(y_test_0, y_pred_hybrid, average='weighted')
rec_h  = recall_score(y_test_0,  y_pred_hybrid, average='weighted')
f1_h   = f1_score(y_test_0,     y_pred_hybrid, average='weighted')

y_bin_h = label_binarize(y_test_0, classes=[0,1,2,3,4])
auc_h   = roc_auc_score(y_bin_h, y_pred_proba_hybrid, multi_class='ovr', average='weighted')

CLASS_NAMES = ['Sangat Rendah (1)', 'Rendah (2)', 'Sedang (3)', 'Tinggi (4)', 'Sangat Tinggi (5)']

print('=' * 55)
print('   EVALUASI MODEL – Hybrid CNN + XGBoost')
print('=' * 55)
print(f'  Accuracy  : {acc_h:.4f}  ({acc_h*100:.2f}%)')
print(f'  Precision : {prec_h:.4f}  ({prec_h*100:.2f}%)')
print(f'  Recall    : {rec_h:.4f}  ({rec_h*100:.2f}%)')
print(f'  F1-Score  : {f1_h:.4f}  ({f1_h*100:.2f}%)')
print(f'  ROC-AUC   : {auc_h:.4f}  ({auc_h*100:.2f}%)')
print('=' * 55)
print()
print(classification_report(y_test_0, y_pred_hybrid, target_names=CLASS_NAMES))

### 5.13 Confusion Matrix – Hybrid

In [ ]:
cm_h     = confusion_matrix(y_test_0, y_pred_hybrid)
cm_h_pct = cm_h.astype('float') / cm_h.sum(axis=1)[:, np.newaxis] * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm_h, annot=True, fmt='d', cmap='Purples', ax=axes[0],
            xticklabels=range(1,6), yticklabels=range(1,6),
            linewidths=0.5, linecolor='gray', cbar_kws={'shrink':0.8})
axes[0].set_title('Confusion Matrix – Hybrid CNN+XGB (Count)', fontweight='bold')
axes[0].set_xlabel('Prediksi')
axes[0].set_ylabel('Aktual')

sns.heatmap(cm_h_pct, annot=True, fmt='.1f', cmap='Greens', ax=axes[1],
            xticklabels=range(1,6), yticklabels=range(1,6),
            linewidths=0.5, linecolor='gray', cbar_kws={'shrink':0.8,'label':'%'})
axes[1].set_title('Confusion Matrix – Hybrid CNN+XGB (%)', fontweight='bold')
axes[1].set_xlabel('Prediksi')
axes[1].set_ylabel('Aktual')

plt.suptitle('Analisis Confusion Matrix – Hybrid CNN + XGBoost', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('05_confusion_matrix_hybrid.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.14 Perbandingan: XGBoost vs Hybrid CNN+XGBoost

In [ ]:
# Load hasil XGBoost sebelumnya
xgb_only_model = joblib.load('xgboost_model.pkl')
X_test_orig    = pd.read_csv('X_test.csv')
y_pred_xgb     = xgb_only_model.predict(X_test_orig)

acc_xgb  = accuracy_score(y_test_0, y_pred_xgb)
prec_xgb = precision_score(y_test_0, y_pred_xgb, average='weighted')
rec_xgb  = recall_score(y_test_0,   y_pred_xgb, average='weighted')
f1_xgb   = f1_score(y_test_0,       y_pred_xgb, average='weighted')
proba_xgb = xgb_only_model.predict_proba(X_test_orig)
auc_xgb  = roc_auc_score(y_bin_h, proba_xgb, multi_class='ovr', average='weighted')

cnn_acc_val = cnn_acc  # dari evaluasi CNN standalone tadi

# Tabel perbandingan
compare_df = pd.DataFrame({
    'Model'    : ['XGBoost Only', 'CNN (Standalone)', 'Hybrid CNN + XGBoost'],
    'Accuracy' : [acc_xgb,  cnn_acc_val, acc_h],
    'Precision': [prec_xgb, None,        prec_h],
    'Recall'   : [rec_xgb,  None,        rec_h],
    'F1-Score' : [f1_xgb,   None,        f1_h],
    'ROC-AUC'  : [auc_xgb,  None,        auc_h]
})
print('=== Perbandingan Model ===')
print(compare_df.to_string(index=False))

# Visualisasi perbandingan
metrics_cmp  = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
vals_xgb     = [acc_xgb,  prec_xgb, rec_xgb, f1_xgb,  auc_xgb]
vals_cnn     = [cnn_acc_val, None, None, None, None]
vals_hybrid  = [acc_h, prec_h, rec_h, f1_h, auc_h]

x  = np.arange(len(metrics_cmp))
w  = 0.28

fig, ax = plt.subplots(figsize=(14, 7))
b1 = ax.bar(x - w, vals_xgb,   w, label='XGBoost Only',         color='#3498db', edgecolor='black')
b2 = ax.bar(x,     [cnn_acc_val]+[0]*4, w, label='CNN Standalone (Acc only)', color='#2ecc71', edgecolor='black', alpha=0.7)
b3 = ax.bar(x + w, vals_hybrid, w, label='Hybrid CNN + XGBoost', color='#8e44ad', edgecolor='black')

ax.set_xticks(x)
ax.set_xticklabels(metrics_cmp)
ax.set_ylim(0.80, 1.05)
ax.set_ylabel('Score')
ax.set_title('Perbandingan Performa Model\nXGBoost vs CNN Standalone vs Hybrid CNN+XGBoost',
             fontweight='bold', fontsize=13)
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)

for bars in [b1, b3]:
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.003,
                    f'{h:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

# Anotasi CNN hanya accuracy
ax.text(b2[0].get_x() + b2[0].get_width()/2, cnn_acc_val + 0.003,
        f'{cnn_acc_val:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig('05_perbandingan_model.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.15 Feature Importance – XGBoost Hybrid

> Karena fitur input ke XGBoost sekarang adalah gabungan  
> 256 deep features CNN + fitur tabular asli, kita lihat  
> fitur mana yang paling penting secara keseluruhan.

In [ ]:
# Nama fitur hybrid
cnn_feat_names  = [f'CNN_feat_{i}' for i in range(256)]
orig_feat_names = list(X_test.columns)
all_feat_names  = cnn_feat_names + orig_feat_names

# Gain importance
scores_gain = xgb_hybrid.get_booster().get_score(importance_type='gain')
feat_imp_df = pd.DataFrame(list(scores_gain.items()), columns=['Feature_idx', 'Gain'])

# Map f0, f1, ... ke nama fitur
feat_imp_df['Feature'] = feat_imp_df['Feature_idx'].apply(
    lambda x: all_feat_names[int(x[1:])] if x.startswith('f') else x
)

# Pisahkan CNN vs Tabular features
feat_imp_df['Type'] = feat_imp_df['Feature'].apply(
    lambda x: 'CNN Feature' if x.startswith('CNN_feat') else 'Tabular Feature'
)

# Agregasi: total gain CNN vs Tabular
type_gain = feat_imp_df.groupby('Type')['Gain'].sum()
print('=== Kontribusi Fitur: CNN vs Tabular (Total Gain) ===')
print(type_gain)
print(f'\nProporsi CNN Features  : {type_gain["CNN Feature"]/type_gain.sum()*100:.1f}%')
print(f'Proporsi Tabular Feats : {type_gain["Tabular Feature"]/type_gain.sum()*100:.1f}%')

# Top-20 fitur tabular terpenting di model hybrid
tabular_imp = feat_imp_df[feat_imp_df['Type'] == 'Tabular Feature'].nlargest(20, 'Gain')

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Pie: CNN vs Tabular
axes[0].pie(type_gain.values, labels=type_gain.index,
            autopct='%1.1f%%', colors=['#8e44ad', '#3498db'],
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[0].set_title('Proporsi Kontribusi Fitur\nCNN vs Tabular (Gain)', fontweight='bold')

# Bar: Top tabular features
axes[1].barh(tabular_imp['Feature'][::-1], tabular_imp['Gain'][::-1],
             color='#3498db', edgecolor='black', linewidth=0.5)
axes[1].set_title('Top Tabular Features dalam Hybrid Model\n(Gain Importance)', fontweight='bold')
axes[1].set_xlabel('Gain')

plt.suptitle('Feature Importance – Hybrid CNN + XGBoost', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('05_feature_importance_hybrid.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.16 Dashboard Akhir – Hybrid CNN + XGBoost

In [ ]:
report_h = classification_report(y_test_0, y_pred_hybrid, output_dict=True)
class_m  = pd.DataFrame({
    'Precision': [report_h[str(i)]['precision'] for i in range(5)],
    'Recall'   : [report_h[str(i)]['recall']    for i in range(5)],
    'F1-Score' : [report_h[str(i)]['f1-score']  for i in range(5)]
}, index=[f'Kelas {i+1}' for i in range(5)])

fig = plt.figure(figsize=(20, 9))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# -- Subplot 1: Metrik per kelas
ax1 = fig.add_subplot(gs[0, 0])
x   = np.arange(5)
w   = 0.25
ax1.bar(x-w, class_m['Precision'], w, label='Precision', color='#3498db', edgecolor='black')
ax1.bar(x,   class_m['Recall'],    w, label='Recall',    color='#2ecc71', edgecolor='black')
ax1.bar(x+w, class_m['F1-Score'],  w, label='F1-Score',  color='#e74c3c', edgecolor='black')
ax1.set_xticks(x)
ax1.set_xticklabels([f'K{i+1}' for i in range(5)])
ax1.set_ylim(0.7, 1.05)
ax1.set_title('Metrik per Kelas', fontweight='bold')
ax1.legend(fontsize=8)

# -- Subplot 2: Overall metrics bar
ax2 = fig.add_subplot(gs[0, 1])
metrics_l = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
vals_l    = [acc_h, prec_h, rec_h, f1_h, auc_h]
clrs      = ['#3498db','#2ecc71','#f39c12','#e74c3c','#8e44ad']
bars_ax2  = ax2.bar(metrics_l, vals_l, color=clrs, edgecolor='black')
ax2.set_ylim(0.80, 1.05)
ax2.set_title('Overall Performance\nHybrid CNN+XGB', fontweight='bold')
ax2.tick_params(axis='x', rotation=20)
for bar, v in zip(bars_ax2, vals_l):
    ax2.text(bar.get_x() + bar.get_width()/2, v + 0.003,
             f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')

# -- Subplot 3: Perbandingan Accuracy
ax3 = fig.add_subplot(gs[0, 2])
model_names = ['XGBoost\nOnly', 'CNN\nStandalone', 'Hybrid\nCNN+XGB']
accs_cmp    = [acc_xgb, cnn_acc_val, acc_h]
clrs3       = ['#3498db', '#2ecc71', '#8e44ad']
b3ax3       = ax3.bar(model_names, accs_cmp, color=clrs3, edgecolor='black')
ax3.set_ylim(0.80, 1.05)
ax3.set_title('Perbandingan Accuracy\nAntar Model', fontweight='bold')
for bar, v in zip(b3ax3, accs_cmp):
    ax3.text(bar.get_x() + bar.get_width()/2, v + 0.003,
             f'{v:.3f}', ha='center', fontsize=10, fontweight='bold')

# -- Subplot 4: Confidence Distribution
ax4 = fig.add_subplot(gs[1, 0])
correct_h  = y_pred_hybrid == y_test_0
max_prb_h  = y_pred_proba_hybrid.max(axis=1)
ax4.hist(max_prb_h[correct_h],  bins=20, alpha=0.7, color='#2ecc71', label='Benar', edgecolor='black')
ax4.hist(max_prb_h[~correct_h], bins=20, alpha=0.7, color='#e74c3c', label='Salah', edgecolor='black')
ax4.set_title('Distribusi Confidence\n(Benar vs Salah)', fontweight='bold')
ax4.set_xlabel('Max Probability')
ax4.legend()

# -- Subplot 5: Confusion Matrix Heatmap
ax5 = fig.add_subplot(gs[1, 1])
sns.heatmap(cm_h, annot=True, fmt='d', cmap='Purples', ax=ax5,
            xticklabels=range(1,6), yticklabels=range(1,6),
            linewidths=0.5, cbar=False)
ax5.set_title('Confusion Matrix', fontweight='bold')
ax5.set_xlabel('Prediksi')
ax5.set_ylabel('Aktual')

# -- Subplot 6: CNN vs Tabular Pie
ax6 = fig.add_subplot(gs[1, 2])
ax6.pie(type_gain.values, labels=type_gain.index, autopct='%1.1f%%',
        colors=['#8e44ad','#3498db'],
        startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
ax6.set_title('Kontribusi Fitur\nCNN vs Tabular', fontweight='bold')

plt.suptitle('Dashboard Evaluasi – Hybrid CNN + XGBoost\nDeteksi Kecanduan Smartphone',
             fontsize=15, fontweight='bold', y=1.01)
plt.savefig('05_dashboard_hybrid.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Dashboard akhir disimpan.')

### 5.17 Simpan Model Hybrid

In [ ]:
# Simpan CNN feature extractor
feature_extractor.save('cnn_feature_extractor.h5')
cnn_model.save('cnn_full_model.h5')

# Simpan XGBoost Hybrid
joblib.dump(xgb_hybrid, 'hybrid_cnn_xgb_model.pkl')
joblib.dump(best_params_hybrid, 'best_params_hybrid.pkl')

print('✅ Model Hybrid tersimpan:')
print('   - cnn_feature_extractor.h5  (CNN tanpa head, untuk inferensi)')
print('   - cnn_full_model.h5         (CNN lengkap dengan classification head)')
print('   - hybrid_cnn_xgb_model.pkl  (XGBoost yang dilatih pada deep features)')
print('   - best_params_hybrid.pkl    (Hyperparameter terbaik)')

print(f'\n=== RINGKASAN AKHIR ===')
print(f'  XGBoost Only     : Accuracy = {acc_xgb*100:.2f}%')
print(f'  CNN Standalone   : Accuracy = {cnn_acc_val*100:.2f}%')
print(f'  Hybrid CNN+XGB   : Accuracy = {acc_h*100:.2f}%')
peningkatan = (acc_h - acc_xgb)*100
print(f'  Peningkatan vs XGBoost Only: {peningkatan:+.2f}%')

---
## Cara Inferensi Model Hybrid pada Data Baru

```python
import numpy as np
import joblib
from tensorflow import keras

# Load artifacts
scaler          = joblib.load('minmax_scaler.pkl')
feature_ext     = keras.models.load_model('cnn_feature_extractor.h5')
hybrid_xgb      = joblib.load('hybrid_cnn_xgb_model.pkl')
selected_feats  = joblib.load('selected_features.pkl')

# Input: data mentah 1 baris (dict)
new_data = pd.DataFrame([{
    'App Usage Time (min/day)': 450,
    'Screen On Time (hours/day)': 8.5,
    'Battery Drain (mAh/day)': 2300,
    # ... fitur lainnya ...
}])

# 1. Normalisasi
X_new_scaled = scaler.transform(new_data[selected_feats])

# 2. Reshape untuk CNN
X_new_cnn = X_new_scaled.reshape(-1, X_new_scaled.shape[1], 1)

# 3. Ekstrak deep features
deep_feats = feature_ext.predict(X_new_cnn)

# 4. Gabungkan dengan fitur tabular
X_hybrid = np.hstack([deep_feats, X_new_scaled])

# 5. Prediksi
pred_class = hybrid_xgb.predict(X_hybrid)[0] + 1  # +1 karena 0-indexed
pred_proba = hybrid_xgb.predict_proba(X_hybrid)[0]
print(f'Prediksi Kelas Kecanduan: {pred_class}')
print(f'Probabilitas: {pred_proba}')
```